# 02 — Data Quality & Protocol Conformance

**Input:** `data/interim/fudo_revenue_clean.parquet` (from notebook 01)
**Protocol:** *Protocolo de Comunicación de Datos Transaccionales Fudo–Skalar (2026-04-09)*

This notebook validates:
1. **Dtype conformance** — does each column match the protocol's expected type?
2. **Value-domain checks** — are values within allowed ranges / categories?
3. **Nullability** — does nullability match NOT NULL / NULLABLE contracts?
4. **Grain integrity** — is `(client_id, transaction_date, product)` a unique key?
5. **Multi-product accounts** — verifying that collections are per-product, not aggregated.
6. **Missing protocol columns** — `channel`, `client_segment` gap.

In [1]:
import pandas as pd
import yaml

from fudo.utils.io import INTERIM_DIR, CONFIGS_DIR

# Load cleaned data from notebook 01
df = pd.read_parquet(INTERIM_DIR / "fudo_revenue_clean.parquet")
df["transaction_date"] = df["transaction_date"].apply(
    lambda v: pd.Period(v, freq="M")
)

# Load protocol schema
with open(CONFIGS_DIR / "dataset_schema.yaml") as f:
    SCHEMA = yaml.safe_load(f)

print(f"Shape: {df.shape}")
df.dtypes

Shape: (33959, 7)


client_id                       str
transaction_date          period[M]
currency                        str
collections                 float64
product                         str
pais                            str
fecha_primer_pago    datetime64[us]
dtype: object

## 1. Dtype conformance

Compare each column's actual pandas dtype against what the protocol requires.

The protocol specifies types at the logical level (STRING, DATE, FLOAT64).
After cleaning in notebook 01 we map:

| Protocol type | Acceptable pandas dtypes | Notes |
|---|---|---|
| `STRING` | `object`, `str`, `string` | Pandas 2.2+ returns `str` from pyarrow parquet |
| `DATE (YYYY-MM-DD)` | `datetime64[ns]`, `datetime64[us]` | **Source delivers `period[M]` — non-conformant** |
| `FLOAT64` | `float64` | Numeric revenue |

`transaction_date` is expected to **FAIL** this check. The protocol requires
daily granularity; the source delivers monthly periods.

In [2]:
# Map protocol dtype strings to expected pandas dtype strings.
# In pandas 2.2+ with pyarrow, string columns from parquet are `str` (StringDtype),
# not `object`. Both represent the protocol's STRING type.
PROTOCOL_TO_PANDAS = {
    "str": {"object", "str", "string"},  # any string-like dtype is acceptable
    "float64": {"float64"},
    "datetime64[ns]": {"datetime64[ns]", "datetime64[us]"},
}

canonical_cols = [c for c in SCHEMA["columns"] if c["raw_name"] is not None]

dtype_checks = []
for spec in canonical_cols:
    col = spec["name"]
    acceptable = PROTOCOL_TO_PANDAS.get(spec["dtype"], {spec["dtype"]})
    actual = str(df[col].dtype) if col in df.columns else "MISSING"
    passed = actual in acceptable

    row = {
        "column": col,
        "protocol_dtype": spec["dtype"],
        "expected_pandas": ", ".join(sorted(acceptable)),
        "actual_pandas": actual,
        "pass": passed,
    }

    # Annotate conformance detail from schema if present
    if "conformance" in spec and spec["conformance"].get("status") == "FAIL":
        row["note"] = spec["conformance"]["detail"].strip()
    else:
        row["note"] = ""

    dtype_checks.append(row)

dtype_report = pd.DataFrame(dtype_checks)
dtype_report.style.map(
    lambda v: "background-color: #d9ead3" if v is True
              else "background-color: #f4cccc" if v is False else "",
    subset=["pass"],
)

,column,protocol_dtype,expected_pandas,actual_pandas,pass,note
0,client_id,str,"object, str, string",str,True,
1,transaction_date,datetime64[ns],"datetime64[ns], datetime64[us]",period[M],False,"Protocol requires daily granularity (YYYY-MM-DD). Fudo delivers monthly period (YYYYMM). This forces Skalar to assume uniform distribution or arbitrary date assignment within the month, compromising the single-source-of-truth principle (protocol §4.1)."
2,currency,str,"object, str, string",str,True,
3,collections,float64,float64,float64,True,
4,product,str,"object, str, string",str,True,


### 1.1 Non-conformance: `transaction_date` granularity

The dtype check above correctly **fails** for `transaction_date`.

| Dimension | Protocol (§3.1) | Actual delivery |
|---|---|---|
| Type | `DATE` | Monthly period (`YYYYMM`) |
| Format | `YYYY-MM-DD` | `YYYYMM` with thousand-separator |
| Granularity | Per-transaction (daily) | Pre-aggregated to month |
| Pandas dtype | `datetime64[ns]` | `period[M]` |

**Why this matters — single source of truth (protocol §4.1):**

> *"Los datos transaccionales se originan en Fudo. No existe transferencia de
> custodia ni ventana donde los datos consultados por Skalar divergen de los
> creados por Fudo."*

By delivering monthly aggregates instead of daily transactions, Fudo forces
Skalar into one of two positions:

1. **Accept month-level granularity.** Skalar loses the ability to audit
   individual transactions, apply daily FX rates, or perform intra-month
   cohort assignment. Operationally acceptable for now, but it constrains
   future analysis.

2. **Fabricate a daily date.** Assigning an arbitrary date (e.g., last day of
   month) to each row introduces an **assumption that originates at Skalar,
   not at Fudo**. Any downstream calculation that depends on that fabricated
   date (settlement timing, cohort boundaries, period-close reconciliation)
   is now built on Skalar's interpretation, not Fudo's data. This is a de
   facto transfer of custody — exactly what the protocol was designed to
   prevent.

**Recommendation:** Fudo should deliver `transaction_date` at daily
granularity as specified in the protocol. The `Payment_date` field referenced
in protocol §3.2 exists in Fudo's source system. Until then, Skalar
operates on monthly aggregates and explicitly documents this as a known
deviation.

## 2. Value-domain checks

Validate that values fall within the protocol's allowed domains.

In [3]:
domain_checks = []

for spec in canonical_cols:
    col = spec["name"]
    if col not in df.columns:
        continue

    # Check valid_values (categorical domains)
    if "valid_values" in spec and spec["valid_values"]:
        allowed = set(spec["valid_values"])
        actual_vals = set(df[col].dropna().unique())
        unexpected = actual_vals - allowed
        domain_checks.append({
            "column": col,
            "check": "valid_values",
            "expected": sorted(allowed),
            "violations": sorted(unexpected) if unexpected else None,
            "violation_count": len(df[df[col].isin(unexpected)]) if unexpected else 0,
            "pass": len(unexpected) == 0,
        })

    # Check valid_range (numeric bounds)
    if "valid_range" in spec and spec["valid_range"]:
        lo, hi = spec["valid_range"]
        series = df[col]
        below = series[series < lo].count() if lo is not None else 0
        above = series[series > hi].count() if hi is not None else 0
        domain_checks.append({
            "column": col,
            "check": "valid_range",
            "expected": spec["valid_range"],
            "violations": f"{below} below min, {above} above max",
            "violation_count": below + above,
            "pass": (below + above) == 0,
        })

domain_report = pd.DataFrame(domain_checks)
domain_report.style.map(
    lambda v: "background-color: #d9ead3" if v is True
              else "background-color: #f4cccc" if v is False else "",
    subset=["pass"],
)

,column,check,expected,violations,violation_count,pass
0,currency,valid_values,"['ARS', 'BRL', 'CLP', 'COP', 'MXN', 'USD']",nan,0,True
1,collections,valid_range,"[0, None]","0 below min, 0 above max",0,True
2,product,valid_values,"['Online Ordering', 'Payments', 'Saas']",nan,0,True


## 3. Nullability checks

The protocol defines NOT NULL for `client_id`, `transaction_date`, `currency`,
`collections`. NULLABLE for `product`, `channel`, `client_segment`.

In [4]:
null_checks = []
for spec in SCHEMA["columns"]:
    col = spec["name"]
    if col not in df.columns:
        null_checks.append({
            "column": col,
            "nullable": spec["nullable"],
            "null_count": "N/A — column missing",
            "pass": None,
        })
        continue

    n_null = int(df[col].isnull().sum())
    # NOT NULL columns must have zero nulls
    if not spec["nullable"]:
        passed = n_null == 0
    else:
        passed = True  # nulls are allowed
    null_checks.append({
        "column": col,
        "nullable": spec["nullable"],
        "null_count": n_null,
        "null_pct": round(n_null / len(df) * 100, 2) if n_null else 0.0,
        "pass": passed,
    })

null_report = pd.DataFrame(null_checks)
null_report.style.map(
    lambda v: "background-color: #d9ead3" if v is True
              else "background-color: #f4cccc" if v is False
              else "background-color: #fff2cc" if v is None else "",
    subset=["pass"],
)

,column,nullable,null_count,null_pct,pass
0,client_id,False,0,0.000000,True
1,transaction_date,False,0,0.000000,True
2,currency,False,0,0.000000,True
3,collections,False,0,0.000000,True
4,product,True,0,0.000000,True
5,channel,True,N/A — column missing,nan,None
6,client_segment,True,N/A — column missing,nan,None


## 4. Grain integrity

The expected grain is one row per `(client_id, transaction_date, product)`.
Zero duplicates = the data can serve as a fact table at this grain.

In [5]:
grain_cols = SCHEMA["grain"]
n_rows = len(df)
n_keys = df.groupby(grain_cols).ngroups
n_dupes = n_rows - n_keys

print(f"Grain columns:  {grain_cols}")
print(f"Total rows:     {n_rows:,}")
print(f"Unique keys:    {n_keys:,}")
print(f"Duplicate rows: {n_dupes:,}")
print(f"\nGrain test: {'PASS' if n_dupes == 0 else 'FAIL'}")

if n_dupes > 0:
    dupes = df[df.duplicated(subset=grain_cols, keep=False)].sort_values(grain_cols)
    print(f"\nSample duplicates:")
    display(dupes.head(10))

Grain columns:  ['client_id', 'transaction_date', 'product']
Total rows:     33,959
Unique keys:    33,959
Duplicate rows: 0

Grain test: PASS


## 5. Multi-product accounts — collections granularity

Some accounts have revenue from multiple products in the same month.
This verifies that `collections` is reported **per product**, not summed
at the account level — critical for downstream cohort calculations.

In [6]:
# How many (client_id, month) pairs have multiple products?
products_per_acct_month = (
    df.groupby(["client_id", "transaction_date"])["product"]
    .nunique()
    .rename("n_products")
)

multi = products_per_acct_month[products_per_acct_month > 1]
total_pairs = len(products_per_acct_month)

print(f"Total (client_id, month) pairs: {total_pairs:,}")
print(f"With multiple products:         {len(multi):,} ({len(multi)/total_pairs*100:.1f}%)")
print(f"\nProduct combination breakdown:")
combos = (
    df.groupby(["client_id", "transaction_date"])["product"]
    .apply(lambda x: " + ".join(sorted(x)))
    .value_counts()
)
combos[combos.index.str.contains(r"\+")].reset_index(name="count").rename(
    columns={"product": "combination"}
)

Total (client_id, month) pairs: 30,621
With multiple products:         3,171 (10.4%)

Product combination breakdown:


,combination,count
0,Payments + Saas,2223
1,Online Ordering + Saas,757
2,Online Ordering + Payments + Saas,167
3,Online Ordering + Payments,24


In [7]:
# Show a concrete example: one multi-product account across months
sample_multi_id = multi.index.get_level_values("client_id")[0]
sample = df[df["client_id"] == sample_multi_id].sort_values(
    ["transaction_date", "product"]
)
print(f"Example multi-product account: {sample_multi_id}\n")
sample[["client_id", "transaction_date", "product", "currency", "collections"]]

Example multi-product account: 290115



,client_id,transaction_date,product,currency,collections
767,290115,2026-01,Online Ordering,CLP,37533.80
28712,290115,2026-01,Saas,CLP,38950.00
15397,290115,2026-02,Online Ordering,CLP,24330.40
17140,290115,2026-02,Saas,CLP,41800.00
12425,290115,2026-03,Online Ordering,CLP,19896.03
24314,290115,2026-03,Saas,CLP,41800.00
20127,290115,2026-04,Online Ordering,CLP,10879.09
6700,290115,2026-04,Saas,CLP,41800.00


## 6. Missing protocol columns

The protocol defines 7 columns. We have 5. The missing ones are `channel` and
`client_segment` — both NULLABLE, so their absence doesn't violate NOT NULL
contracts, but it limits Skalar's ability to run segmented cohort analysis.

In [8]:
protocol_cols = {c["name"] for c in SCHEMA["columns"]}
present_cols = set(df.columns) & protocol_cols
missing_cols = protocol_cols - present_cols
extra_in_df = set(df.columns) - protocol_cols

print("Protocol columns present:  ", sorted(present_cols))
print("Protocol columns MISSING:  ", sorted(missing_cols))
print("Extra columns (not in protocol):", sorted(extra_in_df))

Protocol columns present:   ['client_id', 'collections', 'currency', 'product', 'transaction_date']
Protocol columns MISSING:   ['channel', 'client_segment']
Extra columns (not in protocol): ['fecha_primer_pago', 'pais']


## 7. Completeness — zero-revenue rows

756 rows have `collections == 0`. These are accounts that were active (present
in the data) but generated no revenue that month. This is valid — not a data
quality issue — but worth flagging for downstream awareness.

In [9]:
zeros = df[df["collections"] == 0]
print(f"Zero-collections rows: {len(zeros):,} / {len(df):,} ({len(zeros)/len(df)*100:.1f}%)")
print(f"\nBreakdown by product:")
print(zeros["product"].value_counts().to_string())
print(f"\nBreakdown by currency:")
print(zeros["currency"].value_counts().to_string())

Zero-collections rows: 756 / 33,959 (2.2%)

Breakdown by product:
product
Payments    469
Saas        287

Breakdown by currency:
currency
ARS    278
CLP    223
MXN    124
BRL     94
COP     28
USD      9


## Summary

| Check | Result |
|---|---|
| Dtype conformance (5 cols) | **4 PASS, 1 FAIL** (`transaction_date`: `period[M]` vs required `datetime64[ns]`) |
| `transaction_date` granularity | **FAIL** — monthly aggregate vs protocol-required daily. See §1.1 |
| Value domains (currency, product, collections) | PASS |
| Nullability (NOT NULL contracts) | PASS |
| Grain uniqueness `(client_id, month, product)` | PASS (0 duplicates) |
| Collections granularity (per-product, not aggregated) | Confirmed |
| Missing protocol columns (`channel`, `client_segment`) | Absent |
| Zero-revenue rows | 756 rows (2.2%), valid but flagged |

The `transaction_date` non-conformance is the critical finding. It imposes a
structural limitation: any date-level assumption Skalar makes downstream is
Skalar's own construction, not Fudo's data — violating the single-source-of-truth
principle (protocol §4.1).